<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261106/%E3%83%8F%E3%82%A4%E3%83%91%E3%83%BC%E3%83%91%E3%83%A9%E3%83%A1%E3%83%BC%E3%82%BF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ハイパーパラメータチューニング

## 本日の講義の目的

本講義では、機械学習モデルの汎化性能を高めるために用いられる
**ハイパーパラメータチューニング**を学ぶ。


#ハイパーパラメータチューニングの基本

##モデルパラメータとハイパーパラメータ

機械学習では、
**モデルパラメータ**と**ハイパーパラメータ**を適切に定める必要がある。

| 区分 | 意味 | 例 |
|---|---|---|
| モデルパラメータ | データから学習アルゴリズムによって推定される値 | 線形回帰・ロジスティック回帰の係数、切片 |
| ハイパーパラメータ | モデルの複雑さを制御するために学習前に設定する値 | 正則化強度、木の深さ、近傍数、学習率 |

以下の表は、代表的なモデルとそのハイパーパラメータの例である。

| モデル | 代表的なハイパーパラメータ |
|---|---|
| ロジスティック回帰 | `C`、正則化の方法（L1,L2） |
| Ridge 回帰 | `alpha` |
| 決定木 | `max_depth`、`min_samples_leaf` |
| k-NN | `n_neighbors` |
| ランダムフォレスト | `n_estimators`、`max_depth`、`max_features` |
| 勾配ブースティング | `learning_rate`、`n_estimators`、木の深さ |

ロジスティック回帰で行った正則化パラメータの調整は
ハイパーパラメータチューニングの一例である。

##なぜチューニングが必要か

ハイパーパラメータによって **モデルの複雑さ**
が変化する。

- モデルが単純すぎると、学習データの特徴を十分に捉えられず
**過小適合（underfitting）**
が起こる。
- モデルが複雑すぎると、学習データに過度に適合し、未知データに対する性能が低下する
**過学習（overfitting）**
が起こる。

ハイパーパラメータチューニングの目的は
**未知データに対する汎化性能を高めること**である。

##基本的な手順

ハイパーパラメータのチューニングは、以下の流れが基本である。

1. データを**学習用データ**と**テストデータ**に分割する。
2. 学習用データのみで交差検証を行い、ハイパーパラメータを選択する。
3. 最良のハイパーパラメータを用いて学習用データ全体でモデルを再学習する。
4. 最後にテストデータを用いて性能(精度)を評価する。

`GridSearchCV` は、候補となるハイパーパラメータごとに交差検証を行い、
最良の値を選択する機能である。
既定では、
最良の値を用いて**全ての学習用データ**
で学習されたモデルが
 `best_estimator_` に格納される。

> **重要**  
> テストデータでの性能を見ながらハイパーパラメータを調整してはならない。
テストデータは原則として最終確認にのみ用いる。


##前処理とデータリーケージ

欠損値補完、標準化、特徴量選択などの前処理で、データ全体の情報を利用すると、
検証データの情報が学習側に混入することになり、
**データリーケージ**が発生する。

特に、交差検証を行う場合には、各foldの学習部分だけを使って前処理を学習する必要がある。そのため、処理をまとめて `Pipeline` 化を行うことが
有効な方法として採用されている。すなわち、

- 欠損値補完
- 標準化
- One-Hot Encoding
- 予測モデル

この処理全体を `Pipeline` 化して `GridSearchCV` に渡すことで、
各foldの学習データだけから前処理が学習される。


#環境の準備

以下のコードセルを実行し、分析に必要なライブラリを読み込む。


In [ ]:
%%capture
# 日本語表示用ライブラリ
!pip -q install japanize-matplotlib

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import japanize_matplotlib

from scipy.stats import loguniform

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_val_score,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#クロスバリデーション（交差検証）

##クロスバリデーションの必要性

機械学習モデルの性能を評価する最も単純な方法は、
データを学習用データと評価用データに一度だけ分割する**ホールドアウト法**である。
しかし、1回の分割だけでは、どのデータが学習側・評価側に入ったかによって評価値が変動することがある。

特にデータ件数が多くない場合には、偶然「予測しやすいデータ」が評価側に集まったり、
逆に「予測しにくいデータ」が評価側に集まったりする可能性がある。
そのため、1回の分割だけからモデルの性能を判断すると、評価が不安定になる場合がある。

**クロスバリデーション（cross-validation：交差検証）**
は、
学習用データを複数の部分に分割し、学習と検証を複数回繰り返すことで、
未知データに対する性能をより安定して推定する方法である。
クロスバリデーションは最も標準的な
**モデルの評価方法**である。


##k-fold クロスバリデーション

最も代表的な方法が **k-fold cross-validation（k分割交差検証）** である。

全データをほぼ同じ大きさの $k$ 個のグループ（fold）に分割し、
そのうち1つを検証用、
残りの $(k-1)$ 個を合わせたデータを学習用としてモデルを評価する。
この操作を、すべてのfoldが1回ずつ検証用になるよう繰り返す。

例えば5-foldクロスバリデーションでは、次のように5回の学習・評価を行う。

| 評価回 | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|---|---|---|---|---|---|
| 1回目 | 検証 | 学習 | 学習 | 学習 | 学習 |
| 2回目 | 学習 | 検証 | 学習 | 学習 | 学習 |
| 3回目 | 学習 | 学習 | 検証 | 学習 | 学習 |
| 4回目 | 学習 | 学習 | 学習 | 検証 | 学習 |
| 5回目 | 学習 | 学習 | 学習 | 学習 | 検証 |

各回で得られた評価値（例えば平均二乗誤差値など）を

$$
s_1, s_2, \ldots, s_k
$$

とすると、モデルの評価値としては、各評価値の平均

$$
\bar{s}=\frac{1}{k}\sum_{i=1}^{k}s_i
$$

を用いる。

ただし、平均値だけを見るのではなく、fold間の
**ばらつき**
も確認することが重要である。


##k-fold の分割の様子を確認する

以下では、10件のデータを `KFold()` を使い5分割した例である。
各回でどのデータが学習用・検証用になるかを確認する。


In [ ]:
sample_index = np.arange(10)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

split_rows = []
for fold, (train_idx, valid_idx) in enumerate(kf.split(sample_index), start=1):
    split_rows.append(
        {
            "fold": fold,
            "学習用インデックス": train_idx.tolist(),
            "検証用インデックス": valid_idx.tolist(),
        }
    )

pd.DataFrame(split_rows)


`shuffle=True` を指定すると、分割前にデータを並べ替える。さらに `random_state` を固定することで、同じ分割を再現できる。

なお、時系列データのようにデータの順序そのものが意味を持つ場合には、無条件にシャッフルしてはならない。データや課題に適した交差検証方法を選択する必要がある。


##分類では Stratified K-Fold を用いる

分類問題では、各foldに含まれるクラスの割合が大きく変わると、評価値が不安定になることがある。
例えば、生存者20%、死亡者80%のようなデータを単純にランダム分割すると、あるfoldでは生存者がほとんど含まれない可能性がある。

このような場合には **Stratified K-Fold** を用いる。これは、目的変数のクラス比率が各foldでできるだけ維持されるように分割する方法である。

なお、Stratified K-Foldは**クラス不均衡そのものを解消する方法ではない**。あくまで、評価用の各foldでクラス比率が極端に変化することを防ぐ方法である。


##平均値だけでなく、ばらつきも確認する

クロスバリデーションの結果、例えば次のような5個のAccuracyが得られたとする。

0.82, 0.79, 0.85, 0.81, 0.83

このとき、平均Accuracyだけでなく標準偏差も確認する。

平均値が高くても標準偏差が大きい場合、データの分割によって性能が大きく変化している可能性がある。これは、データ件数が少ない、外れ値の影響が大きい、データの構造が不均一である、などの可能性を示唆する。

## `cross_val_score` の主要な引数と役割

scikit-learnでは cross_val_score を用いることで、各foldの評価値を簡単に取得できる。

`cross_val_score` の主要な引数を示す。

- `estimator`: 評価したいモデルのインスタンス（例: `LogisticRegression()`, `Pipeline`）
- `X`: 特徴量データ
- `y`: 目的変数データ
- `cv`: 交差検証の分割戦略。整数を指定すると分割数となる。
`KFold` や `StratifiedKFold` のインスタンスを直接渡すこともできる
- `scoring`: 評価指標の文字列（例: `'accuracy'`, `'f1'`, `'roc_auc'`, `'r2'`, `'neg_mean_squared_error'`）
- `n_jobs`: 並列処理に使うCPUコア数。`-1` を指定すると利用可能なすべてのコアを利用する


##fold の個数

実務では **5-fold** や **10-fold** がよく用いられる。ただし、常に10-foldが最良であるとは限らない。

fold数を増やすと、1回の学習で使用できるデータは増える一方、学習回数も増えるため計算コストが高くなる。

| fold数 | 特徴 |
|---|---|
| 小さい \(k\) | 計算量は少ないが、評価結果が分割の影響を受けやすい場合がある |
| 大きい \(k\) | 各回の学習データは多くなるが、計算量が増える |
| \(k=n\) | Leave-One-Out Cross-Validation（LOOCV）となる |

したがって、fold数はデータ件数、計算時間、モデルの学習コストを考慮して決定する必要がある。

本講義では、特別な理由がない限り **5-fold** を基本とする。


##前処理もfoldの内部で学習する

クロスバリデーションで特に重要なのが、**前処理によるデータリーケージを防ぐこと**である。

例えば、全データを用いて平均値と標準偏差を計算してから標準化し、その後にクロスバリデーションを行うと、検証foldの情報が前処理を通じて学習側に漏れてしまう。

適切な方法は、前処理とモデル推定を `Pipeline` にまとめ、そのPipeline全体を交差検証することである。

```python
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

cross_val_score(
    pipeline,
    X,
    y,
    cv=5
)
```

この方法では、各foldごとに

1. 学習データだけで標準化の平均・標準偏差を計算する。
2. その値で学習データと検証データを変換する。
3. 学習データでモデルを学習する。
4. 検証データで性能を評価する。

という処理が行われる。

したがって、**欠損値補完、標準化、カテゴリ変数のエンコーディング、特徴量選択など、データから学習される前処理はPipelineの中に置く**ことが基本である。


##データ構造に応じた交差検証

すべてのデータに通常のK-Foldを使えるわけではない。データの生成構造を考慮する必要がある。

| データの特徴 | 主な方法 | 考え方 |
|---|---|---|
| 一般的な回帰データ | `KFold` | データをk個に分割する |
| 一般的な分類データ | `StratifiedKFold` | クラス比率をできるだけ維持する |
| 同一人物・同一企業などのグループがある | `GroupKFold` | 同じグループが学習側と検証側に跨がないようにする |
| 時系列データ | `TimeSeriesSplit` | 過去で学習し、未来を検証する |

例えば、同一患者から複数回取得した検査データが存在する場合、同じ患者のデータが学習foldと検証foldの両方に入ると、実際より高い予測性能が得られる可能性がある。この場合には患者IDをグループとして扱う必要がある。

また、時系列問題では未来のデータを使って過去の現象を予測するような分割を行ってはならない。


##クロスバリデーションとハイパーパラメータチューニング

ハイパーパラメータチューニングでは、複数の候補を平等に比較する必要がある。
そのためにクロスバリデーションを利用する。

例えば、ロジスティック回帰の `C` として

$$
0.01,\ 0.1,\ 1,\ 10,\ 100, \ 1000
$$

の6候補に比較に 5-fold クロスバリデーションを行う場合であれば、

$$
6 \text{候補} \times 5 \text{fold} = 30
$$

回のモデル学習と評価が必要になる。

`GridSearchCV` は、この処理を自動的に行う仕組みである。

ハイパーパラメータチューニング全体の流れは次のようになる。

1. データを学習用データと最終テストデータに分割する。
2. 学習用データの内部でクロスバリデーションを行う。
3. 各ハイパーパラメータ候補について平均検証性能を計算する。
4. 最も良い候補を選択する。
5. 選択した設定で学習用データ全体に再学習する。
6. 最後に、保持しておいたテストデータで最終評価を行う。

この考え方が、後述する `GridSearchCV` と `RandomizedSearchCV` の基礎となる。


#### 参考資料
ここまでのまとめとして、以下を参照せよ。

https://scikit-learn.org/stable/modules/cross_validation.html


---
#分類モデルチューニングの実際：Titanic データ
実際にデータを用いながら、ハイパーパラメータのチューニングを説明する。


##データの読み込み

ここでは、seaborn に含まれる Titanic データを用いる。このデータは、1912年のタイタニック号沈没事故に関する乗客情報をまとめたものである。


In [ ]:
data = sns.load_dataset("titanic")
data.head()


##データ構造と欠損値の確認


In [ ]:
data.info()

In [ ]:
data.isnull().sum().sort_values(ascending=False)

Titanic データには891件のレコードと15個の項目があり、
`age`、`embarked`、`deck` などの項目に欠損値が含まれる。

主な項目は以下のとおりである。

| 列名 | 型 | 意味 |
|---|---|---|
| survived | カテゴリ | 生存フラグ（0=死亡、1=生存） |
| pclass | カテゴリ | 乗船クラス（1、2、3） |
| sex | カテゴリ | 性別 |
| age | 数値 | 年齢 |
| sibsp | 数値 | 同乗している兄弟・姉妹・配偶者の数 |
| parch | 数値 | 同乗している親・子供の数 |
| fare | 数値 | 運賃 |
| embarked | カテゴリ | 乗船港（C、Q、S） |
| class | カテゴリ | 乗船クラス。`pclass` とほぼ同じ情報である |
| who | カテゴリ | man、woman、child の区分である |
| adult_male | カテゴリ | 成人男性かどうかを表す |
| deck | カテゴリ | デッキ。欠損値が多い |
| embark_town | カテゴリ | 乗船港。`embarked` と同じ情報である |
| alive | カテゴリ | 生存状況。`survived` と同じ目的変数情報である |
| alone | カテゴリ | 単独乗船かどうかを表す |

本講義では、目的変数を `survived` とし、説明変数には以下を用いる。

- 数値変数：`age`、`fare`、`sibsp`、`parch`
- カテゴリ変数：`pclass`、`sex`、`embarked`

`alive` は目的変数 `survived` と同じ内容であるため説明変数には含めない。
また、`class` と `embark_town` はそれぞれ `pclass`、`embarked` と重複するため
使用しない。
`who`、`adult_male`、`alone` は他の列から派生した情報を含むため、
ここでは説明を簡潔にする目的で使用しない。


##学習用データとテストデータへの分割

まず初めに、最終評価用のテストデータを20%確保する。
分類では、目的変数のクラス比率が分割後も大きく変わらないように `stratify=y` を指定する。


In [ ]:
numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["pclass", "sex", "embarked"]

X = data[numeric_features + categorical_features]
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("学習データ:", X_train.shape)
print("テストデータ:", X_test.shape)
print("\n学習データの生存率:", y_train.mean())
print("テストデータの生存率:", y_test.mean())


##前処理を Pipeline に組み込む

数値変数には以下の処理を行う。

1. 欠損値を中央値で補完する。
2. `StandardScaler` で標準化する。

カテゴリ変数には以下の処理を行う。

1. 欠損値を最頻値で補完する。
2. One-Hot Encoding を行う。


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

clf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000)), #C=1.0が規定値で用いられる
    ]
)


##チューニング前の基準性能

まず、`C=1.0` の既定設定を用いたロジスティック回帰を、学習用データ内の5-fold交差検証で評価する。

ここでは説明を簡潔にするため評価指標として Accuracy を用いる。ただし、クラス不均衡や誤分類コストが重要な問題では、Precision、Recall、F1、ROC-AUC などから目的に適した指標を選ぶ必要がある。


In [ ]:
#初期化
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_scores = cross_val_score(
    clf_pipeline,
    X_train,
    y_train,
    cv=cv_clf,
    scoring="accuracy",
)

print("各foldのAccuracy:", np.round(baseline_scores, 3))
print("平均Accuracy:", baseline_scores.mean().round(3))
print("標準偏差:", baseline_scores.std().round(3))


#ロジスティック回帰のハイパーパラメータ `C`

##`C` と正則化

scikit-learn の `LogisticRegression` では、既定で正則化が用いられる。`C` は**正則化強度の逆数**である。

- `C` が小さいほど正則化が強くなる。
- `C` が大きいほど正則化が弱くなる。
- 既定値は `C=1.0` である。

したがって、「大きな `C` ほど常に性能が高い」「小さな `C` ほど常に性能が高い」という関係はない。データに応じて適切な値を交差検証で選択する必要がある。

探索範囲は一律に0.1〜100などと決めるのではなく、まず桁を広く変えた候補を試すことが有効である。ここでは対数スケールで以下を探索する。

$$
10^{-3}, 10^{-2.5}, \ldots, 10^3
$$

このような候補は次のように簡単に作成することができる。

In [ ]:
np.logspace(-3, 3, 13)

##GridSearchCV による探索

`Pipeline` の内部にあるモデルのパラメータを指定する場合は、`モデル名__パラメータ名` の形式を用いる。

`clf_pipeline` を以下
```
clf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=1000)), #C=1.0が規定値で用いられる
    ]
)
```
のように定義した際に用いたモデル名 `model` を使い、
本例では `model__C` となる。


In [ ]:
param_grid = {
    "model__C": np.logspace(-3, 3, 13)
}

grid_search_clf = GridSearchCV(
    estimator=clf_pipeline,
    param_grid=param_grid,
    cv=cv_clf,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)

grid_search_clf.fit(X_train, y_train)

print("最良のパラメータ:", grid_search_clf.best_params_)
print("最良の平均CV Accuracy:", round(grid_search_clf.best_score_, 3))


`GridSearchCV` は既定で `refit=True` であるため、最良のハイパーパラメータを用いて学習用データ全体に再適合したモデルが `best_estimator_` に保存される。したがって `best_estimator_` に対して改めて `fit()` を呼ぶ必要はない。


##探索結果の確認

グリッドサーチの結果は `.cv_results_` に
辞書として格納される。
テストデータの平均と標準偏差がそれぞれ
`mean_test_score` と `std_test_score` である。

In [ ]:
grid_search_clf.cv_results_.keys()

平均スコアだけでなく、候補ごとのばらつきも確認することが重要である。

In [ ]:
cv_results_clf = pd.DataFrame(grid_search_clf.cv_results_)

result_clf = cv_results_clf[
    ["param_model__C", "mean_test_score", "std_test_score", "mean_train_score"]
].copy()

result_clf = result_clf.sort_values("param_model__C")
result_clf


In [ ]:
x = result_clf["param_model__C"].astype(float)
y_mean = result_clf["mean_test_score"]
y_std = result_clf["std_test_score"]

plt.figure(figsize=(9, 5))
plt.errorbar(x, y_mean, yerr=y_std, marker="o", capsize=3)
plt.xscale("log")
plt.xlabel("C")
plt.ylabel("交差検証 Accuracy")
plt.title("ロジスティック回帰：C と交差検証性能")
plt.grid(True)
plt.show()


##最終テストデータでの評価

チューニングが完了した後に、初めてテストデータを用いて最終評価を行う。


In [ ]:
best_clf = grid_search_clf.best_estimator_

y_pred = best_clf.predict(X_test)
y_prob = best_clf.predict_proba(X_test)[:, 1]

metrics_clf = pd.Series(
    {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    }
)

metrics_clf.round(3)


In [ ]:
cm = confusion_matrix(y_test, y_pred)

pd.DataFrame(
    cm,
    index=["実際: 死亡(0)", "実際: 生存(1)"],
    columns=["予測: 死亡(0)", "予測: 生存(1)"],
)


##係数の確認

ロジスティック回帰の係数の正負は、他の変数を固定したときに目的変数が1になる方向との関係を示す。

- 係数が正：値が大きいほど、生存（1）の対数オッズが高くなる方向である。
- 係数が負：値が大きいほど、生存（1）の対数オッズが低くなる方向である。

ただし、係数は因果効果を意味しない。また、One-Hot Encodingされたカテゴリ変数では基準カテゴリとの比較として解釈する必要がある。


In [ ]:
feature_names = best_clf.named_steps["preprocess"].get_feature_names_out()
coef = best_clf.named_steps["model"].coef_[0]

coef_df = (
    pd.DataFrame({"feature": feature_names, "coefficient": coef})
    .sort_values("coefficient")
)

coef_df

#Grid Search と Random Search

ハイパーパラメータのチューニングでは、
大きく分けて2つの戦略がある。

## Grid Search

Grid Search は、あらかじめ指定した候補の**すべての組合せ**を評価する方法である。

**長所**
- 動作が分かりやすい。
- 候補数が少なければ網羅的に比較できる。

**短所**
- ハイパーパラメータの数や候補数が増えると、組合せ数が急増する。

## Random Search

Random Search は、指定した分布から一定回数だけ候補をランダムに抽出して評価する方法である。

**長所**
- 探索空間が大きい場合に、計算予算を制御しやすい。
- 重要でないハイパーパラメータの全候補を網羅する必要がない。

**短所**
- 同じ計算回数では、最良点を必ず調べる保証はない。

ハイパーパラメータが1つだけで候補数も少ない場合はGrid Searchで十分である。多数のハイパーパラメータを同時に調整する場合はRandom Searchが有効である。


##RandomizedSearchCV の例

以下は、ロジスティック回帰の `C` を連続的な対数一様分布から探索する例である。


In [ ]:
random_search_clf = RandomizedSearchCV(
    estimator=clf_pipeline,
    param_distributions={
        "model__C": loguniform(1e-4, 1e4)
    },
    n_iter=20,
    cv=cv_clf,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)

random_search_clf.fit(X_train, y_train)

print("最良のパラメータ:", random_search_clf.best_params_)
print("最良の平均CV Accuracy:", round(random_search_clf.best_score_, 3))


#回帰モデルのチューニング：Tips データ

##データの読み込み

ここでは seaborn の Tips データを用い、チップ金額 `tip` を予測する回帰モデルを作成する。


In [ ]:
df_tips = sns.load_dataset("tips")
df_tips.head()


In [ ]:
df_tips.info()


主な項目は以下のとおりである。

| 項目 | 型 | 説明 |
|---|---|---|
| total_bill | 数値 | 食事代の総額 |
| tip | 数値 | チップ金額 |
| sex | カテゴリ | 性別 |
| smoker | カテゴリ | 喫煙者かどうか |
| day | カテゴリ | 曜日 |
| time | カテゴリ | 昼食か夕食か |
| size | 数値 | グループの人数 |

- 目的変数：`tip`
- 説明変数：`total_bill`、`size`、`sex`、`smoker`、`day`、`time`


##学習用データと最終テストデータへの分割


In [ ]:
reg_numeric_features = ["total_bill", "size"]
reg_categorical_features = ["sex", "smoker", "day", "time"]

X_reg = df_tips[reg_numeric_features + reg_categorical_features]
y_reg = df_tips["tip"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42,
)

print("学習データ:", X_train_reg.shape)
print("最終テストデータ:", X_test_reg.shape)


##回帰用 Pipeline の作成

回帰でも、標準化やOne-Hot Encodingは交差検証の各foldの内部で行う。


In [ ]:
reg_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

reg_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

reg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", reg_numeric_transformer, reg_numeric_features),
        ("cat", reg_categorical_transformer, reg_categorical_features),
    ]
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocess", reg_preprocessor),
        ("model", Ridge(alpha=1.0)),
    ]
)


#回帰のハイパーパラメータ `alpha`

##`alpha` と正則化

Ridge 回帰はL2正則化を行う線形回帰である。`alpha` は正則化の強さを表す。

- `alpha` が小さいほど正則化は弱い。
- `alpha` が大きいほど正則化は強い。

ロジスティック回帰の `C` は「正則化強度の逆数」であるため、方向が逆である点に注意が必要である。

- LogisticRegression:  $C$ を増加させると正則化が弱くなる
- Ridge: $\alpha$ を増加させると正則化が強くなる


##チューニング前の基準性能

ここでは決定係数 $R^2$ を主な評価指標とする。
$R^2$ は1に近いほど良いが、0未満になることもある。


In [ ]:
cv_reg = KFold(n_splits=5, shuffle=True, random_state=42)

baseline_ridge_scores = cross_val_score(
    ridge_pipeline,
    X_train_reg,
    y_train_reg,
    cv=cv_reg,
    scoring="r2",
)

print("各foldのR2:", np.round(baseline_ridge_scores, 3))
print("平均R2:", baseline_ridge_scores.mean().round(3))
print("標準偏差:", baseline_ridge_scores.std().round(3))


##GridSearchCV による `alpha` の探索


In [ ]:
param_grid_ridge = {
    "model__alpha": np.logspace(-3, 3, 13)
}

grid_search_ridge = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid_ridge,
    cv=cv_reg,
    scoring="r2",
    n_jobs=-1,
    return_train_score=True,
)

grid_search_ridge.fit(X_train_reg, y_train_reg)

print("最良のパラメータ:", grid_search_ridge.best_params_)
print("最良の平均CV R2:", round(grid_search_ridge.best_score_, 3))


In [ ]:
cv_results_ridge = pd.DataFrame(grid_search_ridge.cv_results_)

result_ridge = cv_results_ridge[
    ["param_model__alpha", "mean_test_score", "std_test_score", "mean_train_score"]
].copy()

result_ridge = result_ridge.sort_values("param_model__alpha")
result_ridge


In [ ]:
x = result_ridge["param_model__alpha"].astype(float)
y_mean = result_ridge["mean_test_score"]
y_std = result_ridge["std_test_score"]

plt.figure(figsize=(9, 5))
plt.errorbar(x, y_mean, yerr=y_std, marker="o", capsize=3)
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("交差検証 R2")
plt.title("Ridge回帰：alpha と交差検証性能")
plt.grid(True)
plt.show()


##最終テストデータでの評価


In [ ]:
best_ridge = grid_search_ridge.best_estimator_

y_pred_reg = best_ridge.predict(X_test_reg)

r2 = r2_score(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))

print("Test R2:", round(r2, 3))
print("Test RMSE:", round(rmse, 3))


#Elastic Net への拡張

Elastic Net は、L1正則化とL2正則化を組み合わせた線形回帰モデルである。

主なハイパーパラメータは以下である。

- `alpha`：正則化全体の強さ。0以上の数値を指定する。
- `l1_ratio`：L1正則化とL2正則化の混合比率。0以上1以下の数値を指定する。
    - `l1_ratio=1` ではL1正則化のみとなる。
    - `l1_ratio=0` ではL2正則化のみとなる。
    - 0と1の間では両者を組み合わせる。

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html#sklearn.linear_model.ElasticNet


In [ ]:
elastic_pipeline = Pipeline(
    steps=[
        ("preprocess", reg_preprocessor),
        ("model", ElasticNet(max_iter=10000)),
    ]
)

param_grid_elastic = {
    "model__alpha": np.logspace(-3, 1, 9),
    "model__l1_ratio": [0.1, 0.5, 0.9, 1.0],
}

grid_search_elastic = GridSearchCV(
    estimator=elastic_pipeline,
    param_grid=param_grid_elastic,
    cv=cv_reg,
    scoring="r2",
    n_jobs=-1,
)

grid_search_elastic.fit(X_train_reg, y_train_reg)

print("最良のパラメータ:", grid_search_elastic.best_params_)
print("最良の平均CV R2:", round(grid_search_elastic.best_score_, 3))


##Ridge と Elastic Net の比較

モデルを比較する場合も、テストデータを見ながら選択するのではなく、学習用データ内の同じ交差検証条件で比較する。

以下は、今回の学習用データに対する最良の交差検証スコアの比較である。


In [ ]:
comparison = pd.DataFrame(
    {
        "model": ["Ridge", "ElasticNet"],
        "best_cv_r2": [
            grid_search_ridge.best_score_,
            grid_search_elastic.best_score_,
        ],
    }
)

comparison


##`RidgeCV` や `ElasticNetCV` について

scikit-learn には `RidgeCV`、`ElasticNetCV`、`LogisticRegressionCV` など、交差検証を内蔵したモジュールも用意されている。

ただし、欠損値補完・標準化・One-Hot Encodingなど、データから学習する前処理がある場合には注意が必要である。前処理も各foldの学習データだけで学習させる必要があるため、本講義では**前処理とモデルを一つの `Pipeline` にし、そのPipeline全体を `GridSearchCV` に渡す方法**を基本とする。


#ハイパーパラメータチューニングで注意すべき点

##評価指標を先に決める

最良のハイパーパラメータは、評価指標によって変わり得る。

**分類の例**
- Accuracy：全体として何割正しく分類したか。
- Precision：陽性と予測したもののうち、実際に陽性だった割合。
- Recall：実際の陽性のうち、正しく陽性と判定できた割合。
- F1：PrecisionとRecallの調和平均。
- ROC-AUC：判定閾値を変化させたときの識別性能を評価する。

**回帰の例**
- $R^2$：説明できた分散の割合を表す。
- RMSE：大きな誤差を比較的強く評価する。
- MAE：誤差の絶対値を平均し、外れ値の影響がRMSEより小さい。

業務上重要な誤りを考え、チューニング前に評価指標を決める必要がある。


##探索範囲は対数スケールも活用する

正則化パラメータや学習率などは、0.01、0.1、1、10、100のように**桁を変えて探索**すると全体傾向を把握しやすい。

最初から狭い範囲を細かく探索すると、真に良い領域を見逃す可能性がある。一般には以下の手順が有効である。

1. 広い範囲を粗く探索する。
2. 良い領域を絞り込む。
3. 必要であれば、その周辺を細かく再探索する。


##交差検証方法はデータ構造に合わせる

単純なランダムなk-fold交差検証の利用には注意が必要である。

- 分類でクラス比率を保ちたい場合：`StratifiedKFold`
- 同一顧客・同一患者などのグループを跨いで分割してはいけない場合：`GroupKFold`
- 時系列データで未来情報を過去の学習に混ぜてはいけない場合：`TimeSeriesSplit`

データの生成過程に合った分割方法を選択することが重要である。


##計算コストも考慮する

候補数が増えると、必要なモデル学習回数も増える。
例えば、5-fold交差検証で100通りの候補を調べる場合、概ね

$$
100 \times 5 = 500
$$

回の学習が必要になる。

したがって、探索方法は精度だけでなく計算時間も考慮して設計する必要がある。`n_jobs=-1` を指定すると、利用可能なCPUコアを並列利用できる場合がある。


#【まとめ】基本手順

ハイパーパラメータチューニングは、次の順序で実行する。

1. **問題設定を明確にする。**
   - 分類か回帰か
   - どの誤差を重視するか
1. **最終テストデータを先に分離する。**
1. **前処理とモデルをPipeline化する。**
1. **調整対象となるハイパーパラメータを選ぶ。**
1. **探索範囲を設定する。**
1. **Grid SearchやRandom Searchを実行する。**
1. **CVスコアだけでなく、ばらつきや計算時間も確認する。**
1. **必要に応じて探索範囲を絞って再探索する。**
1. **最後にテストデータで一度だけ評価する。**


# 14. 参考資料

- scikit-learn: Model selection and evaluation  
  https://scikit-learn.org/stable/model_selection.html
- scikit-learn: GridSearchCV  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
- scikit-learn: RandomizedSearchCV  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html
- scikit-learn: Pipeline  
  https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html
- scikit-learn: LogisticRegression  
  https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- scikit-learn: Ridge  
  https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
- scikit-learn: Cross-validation  
  https://scikit-learn.org/stable/modules/cross_validation.html
